# Late Arriving Events & Windowing

In streaming pipelines, events frequently arrive out of order — network delays,
mobile devices going offline, batch uploads from edge systems. Aggregating on
**processing time** (when the event was received) instead of **event time** (when it
actually happened) assigns late events to the wrong time window, silently corrupting
your metrics.

This notebook simulates the problem using PySpark in batch mode (no Kafka required)
and demonstrates the watermark concept that Spark Structured Streaming uses in
production.

In [ ]:
!pip install pyspark matplotlib --quiet

## Generate Synthetic Click Events

10 000 click events spread over a 2-hour window:
- **80%** arrive on time (processing delay 0–60 seconds).
- **20%** arrive late (processing delay 5–30 minutes).

Each event has both `event_time` (when the click happened) and `processing_time`
(when the server received it).

In [ ]:
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

random.seed(42)
np.random.seed(42)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("LateArrivingEvents")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

N = 10_000
base_time = datetime(2024, 6, 15, 10, 0, 0)  # 10:00 AM

events = []
for _ in range(N):
    # Event time: uniformly spread over 2 hours
    event_time = base_time + timedelta(seconds=random.randint(0, 7200))

    # Processing delay
    if random.random() < 0.80:
        delay_sec = random.randint(0, 60)           # on-time
    else:
        delay_sec = random.randint(300, 1800)        # late: 5-30 minutes

    processing_time = event_time + timedelta(seconds=delay_sec)

    events.append({
        "user_id": random.randint(1, 500),
        "page": random.choice(["home", "product", "cart", "checkout", "search"]),
        "event_time": event_time,
        "processing_time": processing_time,
    })

events_pdf = pd.DataFrame(events)
events_pdf["lateness_sec"] = (events_pdf["processing_time"] - events_pdf["event_time"]).dt.total_seconds()

events_df = spark.createDataFrame(
    events_pdf[["user_id", "page", "event_time", "processing_time"]]
)

print(f"Total events: {N}")
print(f"On-time (< 60s delay):  {(events_pdf['lateness_sec'] <= 60).sum()}")
print(f"Late (5-30 min delay):  {(events_pdf['lateness_sec'] > 60).sum()}")
events_df.show(5, truncate=False)

## Visualize the Lateness Distribution

The bimodal distribution clearly shows the two populations:
on-time events clustered near 0, and late events spread across 5–30 minutes.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(events_pdf["lateness_sec"] / 60, bins=60, color="steelblue", edgecolor="white")
ax.axvline(x=1, color="green", linestyle="--", label="1 min (on-time threshold)")
ax.axvline(x=10, color="red", linestyle="--", label="10 min (watermark example)")
ax.set_xlabel("Lateness (minutes)")
ax.set_ylabel("Event count")
ax.set_title("Distribution of Event Lateness")
ax.legend()
plt.tight_layout()
plt.show()

## Anti-Pattern: Tumbling Windows on Processing Time

Grouping by `window(processing_time, '5 minutes')` assigns each event to the window
when it was **received**, not when it **happened**. Late events get bucketed into
later windows, inflating those counts and deflating the window where the event
actually belongs.

In [ ]:
# BAD: window on processing_time
proc_time_windows = (
    events_df
    .groupBy(F.window("processing_time", "5 minutes"))
    .agg(F.count("*").alias("event_count"))
    .orderBy("window")
    .withColumn("window_start", F.col("window.start"))
    .select("window_start", "event_count")
)

# GOOD: window on event_time (ground truth)
event_time_windows = (
    events_df
    .groupBy(F.window("event_time", "5 minutes"))
    .agg(F.count("*").alias("event_count"))
    .orderBy("window")
    .withColumn("window_start", F.col("window.start"))
    .select("window_start", "event_count")
)

proc_pdf = proc_time_windows.toPandas().rename(columns={"event_count": "proc_time_count"})
event_pdf = event_time_windows.toPandas().rename(columns={"event_count": "event_time_count"})

comparison = event_pdf.merge(proc_pdf, on="window_start", how="outer").fillna(0).sort_values("window_start")
comparison["difference"] = comparison["proc_time_count"] - comparison["event_time_count"]

print("Side-by-side: event_time windows (correct) vs processing_time windows (wrong)")
display(comparison.head(30))

print(f"\nWindows where counts differ: {(comparison['difference'] != 0).sum()} / {len(comparison)}")
print("The processing_time approach over-counts some windows and under-counts others.")

## Correct Approach: Event-Time Windows

Using `window(event_time, '5 minutes')` assigns events to the window when they
**actually happened**, regardless of when they arrived. This gives accurate per-window
counts.

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

# Filter to the core 2-hour range for clean visualization
mask = (comparison["window_start"] >= base_time) & (
    comparison["window_start"] < base_time + timedelta(hours=2, minutes=30)
)
viz = comparison[mask].copy()
viz["minute"] = (viz["window_start"] - base_time).dt.total_seconds() / 60

axes[0].bar(viz["minute"], viz["event_time_count"], width=4, color="steelblue", label="Event Time")
axes[0].set_title("Event-Time Windows (Correct)")
axes[0].set_xlabel("Minutes since start")
axes[0].set_ylabel("Events per window")

axes[1].bar(viz["minute"], viz["proc_time_count"], width=4, color="coral", label="Processing Time")
axes[1].set_title("Processing-Time Windows (Wrong)")
axes[1].set_xlabel("Minutes since start")

plt.tight_layout()
plt.show()

print("The processing-time chart has inflated counts in later windows (where late events landed).")

## The Watermark Trade-Off

In true streaming, you can't wait forever for late events — at some point you need
to finalize a window and emit results. A **watermark** defines how long to wait:

- Watermark = 10 minutes: accept events up to 10 min late, drop anything later.
- Shorter watermark = lower latency, but more data loss.
- Longer watermark = higher completeness, but higher memory usage and latency.

Let's simulate the trade-off.

In [ ]:
# Simulate different watermark thresholds
watermarks_min = [1, 5, 10, 15, 20, 30]
results = []

for wm in watermarks_min:
    wm_sec = wm * 60
    kept = (events_pdf["lateness_sec"] <= wm_sec).sum()
    dropped = N - kept
    results.append({
        "watermark_minutes": wm,
        "events_kept": kept,
        "events_dropped": dropped,
        "pct_retained": f"{kept / N * 100:.1f}%",
    })

wm_df = pd.DataFrame(results)
print("Watermark trade-off: completeness vs latency")
display(wm_df)

print("\nChoose your watermark based on your SLA:")
print("  - Dashboard refresh every 5 min? Use watermark = 5-10 min.")
print("  - Daily aggregates? Use watermark = 30-60 min or no watermark (batch).")
print("  - Real-time alerting? Use watermark = 1-2 min, accept some data loss.")

## Production Syntax: Spark Structured Streaming with Watermark

In a real streaming pipeline reading from Kafka or Kinesis, the code looks like this:

```python
# Read from Kafka
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "click_events")
    .load()
)

# Parse and apply watermark on event_time
parsed = (
    stream_df
    .select(F.from_json(F.col("value").cast("string"), schema).alias("data"))
    .select("data.*")
    .withWatermark("event_time", "10 minutes")  # <-- the key line
    .groupBy(F.window("event_time", "5 minutes"))
    .agg(F.count("*").alias("click_count"))
)

# Write results
query = (
    parsed.writeStream
    .outputMode("update")
    .format("delta")
    .option("checkpointLocation", "/checkpoints/clicks")
    .start("/output/click_counts")
)
```

The `.withWatermark("event_time", "10 minutes")` tells Spark:
- Use `event_time` for windowing.
- Keep window state open for 10 minutes of lateness.
- After the watermark passes, discard late events and free memory.

In [ ]:
spark.stop()

## Takeaways

| Approach | Accuracy | Latency | Trade-Off |
| :--- | :--- | :--- | :--- |
| Processing-time windows | Low — late events in wrong buckets | Lowest | Simple but incorrect for most use cases |
| Event-time windows (no watermark) | Perfect | Highest (infinite state) | Only feasible in batch mode |
| Event-time + watermark | High (tunable via threshold) | Tunable | The production standard for streaming |

**Golden rule:** always aggregate on **event time**. Use processing time only for
operational metrics ("how fast is our pipeline?"), never for business metrics
("how many clicks per hour?").